In [ ]:
import pandas as pd
import torch
from utils_legacy import Autoencoder, cargar_modelo, crear_datasets_proporcionales
import numpy as np
import json
import os

# --- Carga de datos y configuración del dispositivo ---
#df = pd.read_csv("data/diabetes_binary_health_indicators_BRFSS2015.csv")
df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")

#list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_binary", cols_estandarizar=["BMI", "MentHlth", "PhysHlth", "Age", "Education", "Income"])
list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_012", cols_estandarizar=["BMI", "MentHlth", "PhysHlth", "Age", "Education", "Income"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo usado: {device}")
if device.type == "cuda":
    print(f"Nombre GPU: {torch.cuda.get_device_name(0)}")
x0, x1, x2, x3 = list_x_test
print(resumen_df)


Dispositivo usado: cuda
Nombre GPU: NVIDIA GeForce GTX 970
   Proporción  Positivos  Negativos  Total  % Positivos  % Negativos
0        0.00          0       9262   9262         0.00       100.00
1        0.10        926       8336   9262        10.00        90.00
2        0.25       2315       6947   9262        24.99        75.01
3        0.50       4631       4631   9262        50.00        50.00


In [15]:
#modelo = cargar_modelo("models/autoencoder0.pth")
modelo = cargar_modelo("models/autoencoder_00-54_06-11-25_0.pth")

ejemplo = torch.tensor(x0[0], dtype=torch.float32).to(device)
modelo.eval()
with torch.no_grad():
    reconstruido = modelo(ejemplo)

original = ejemplo.cpu().numpy()
# prediccion = np.round(reconstruido.cpu().numpy())
#prediccion = np.round(np.abs(reconstruido.cpu().numpy()))
prediccion = np.round(np.abs(reconstruido.cpu().numpy()))

aciertos = original == prediccion 

cantidad_aciertos = np.sum(aciertos)

total_valores = original.size
print(f"--- Comparación de Reconstrucción ---")
print(f"{cantidad_aciertos}/{total_valores} valores reconstruidos correctamente.")
print(f"Aciertos: {100 * cantidad_aciertos / total_valores:.2f}%")

print("\nArray original")
print(original)
print("Array reconstruido")
print(prediccion)

Modelo cargado correctamente en cuda.
--- Comparación de Reconstrucción ---
13/21 valores reconstruidos correctamente.
Aciertos: 61.90%

Array original
[ 1.          1.          1.          1.7579322   1.          0.
  0.          0.          0.          1.          0.          1.
  0.          5.          1.9985882   1.2339963   1.          0.
  0.31689945 -1.0655925  -1.4744844 ]
Array reconstruido
[1. 1. 1. 0. 1. 0. 0. 1. 0. 1. 0. 1. 0. 5. 4. 1. 1. 1. 0. 1. 2.]


In [ ]:
from utils_legacy import evaluar_reconstruccion

#df = pd.read_csv("data/diabetes_binary_health_indicators_BRFSS2015.csv")
df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")

# Creamos los conjuntos para el entrenamiento
#list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_binary", cols_estandarizar=["BMI", "MentHlth", "PhysHlth", "Age", "Education", "Income"])
list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_012", cols_estandarizar=["BMI", "MentHlth", "PhysHlth", "Age", "Education", "Income"])

x0, x1, x2, x3 = list_x_test
y0, y1, y2, y3 = list_y_test

mascara_no = y0 == 0
x0_no= x0[mascara_no]
y0_no = y0[mascara_no]

modelo = cargar_modelo("models/autoencoder0.pth")

original, prediccion, diferencias = evaluar_reconstruccion(modelo, x0_no, device, True)

# epsilon = np.max(diferencias) # -> 0% de FP pero 0% de TN
# epsilon = np.mean(diferencias) # -> 48% de FP pero 70% de TN
epsilon = np.mean(diferencias) + 2 * np.std(diferencias )# -> 2.5% de FP pero 5.4% de TN

# probar en datos normales
cant_sup_epsilon = np.sum(diferencias > epsilon)
total_nom = len(diferencias)
porcentaje_nom = 100 * cant_sup_epsilon / total_nom

print(f"Epsilon (solo y0 == 0): {float(epsilon):.2f} para {len(diferencias)} registros")
print(f"De los datos normales el {round(porcentaje_nom, 2)}% son detectados como anomalías aunque no lo sean (FP)")

Modelo cargado correctamente en cuda.
Epsilon (solo y0 == 0): 5.75 para 204441 registros
De los datos normales el 4.41% son detectados como anomalías aunque no lo sean (FP)


c:\entornos-gpu\anomaly-detection-with-autoencoder\utils.py:109: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modelo = torch.load(ruta, map_location=dispositivo)


In [4]:
mascara_si = y0 == 1
x0_si= x0[mascara_si]
y0_si = y0[mascara_si]

# Convertir a tensor
x_tensor_anom = torch.tensor(x0_si, dtype=torch.float32).to(device)

# Evaluar el modelo
modelo.eval()
with torch.no_grad():
    reconstruido_anom = modelo(x_tensor_anom)

# Convertir a numpy y redondear
original_anom = x_tensor_anom.cpu().numpy()
prediccion_anom = np.round(np.abs(reconstruido_anom.cpu().numpy()))

# Calcular diferencias fila a fila
diferencias_anom = np.linalg.norm(original_anom - prediccion_anom, axis=1)

# Contar cuántas superan epsilon
n_superior_epsilon = np.sum(diferencias_anom > epsilon)
total_anom = len(diferencias_anom)
porcentaje = 100 * n_superior_epsilon / total_anom

promedio = np.mean(diferencias_anom)
mediana = np.median(diferencias_anom)
minimo = np.min(diferencias_anom)
maximo = np.max(diferencias_anom)

# Imprimir resultados
print(f"{n_superior_epsilon}/{total_anom} registros de y0==1 superan epsilon ({porcentaje:.2f}%)")
print(f"Diferencia promedio: {promedio:.6f}")
print(f"Diferencia mediana : {mediana:.6f}")
print(f"Diferencia mínima  : {minimo:.6f}")
print(f"Diferencia máxima  : {maximo:.6f}")


363/4631 registros de y0==1 superan epsilon (7.84%)
Diferencia promedio: 3.242611
Diferencia mediana : 2.868377
Diferencia mínima  : 0.496763
Diferencia máxima  : 9.685409
